## 02 · Embedding, BM25 ve hibrit getirme karşılaştırması (CPU, internetsiz)
Kapalı ortamı taklit etmek için paketle gelen WordLlama embedding modeli kullanılıyor.

In [1]:
# Ortam hazırlığı: Colab'da repo klonlanır, yerelde notebooks/ klasöründen proje köküne geçilir
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("agentic-rag-turkish-docs").exists() and not Path("README.md").exists():
    !git clone -q https://github.com/alimdemir/agentic-rag-turkish-docs.git
if IN_COLAB and Path("agentic-rag-turkish-docs").exists():
    %cd agentic-rag-turkish-docs
    !pip -q install -r requirements.txt
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("çalışma klasörü:", Path.cwd().name)

çalışma klasörü: 03-agentic-rag-turkish-docs


In [2]:
import logging, warnings
logging.getLogger("faiss").setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from rag_utils import load_chunks, WordLlamaEmbeddings

chunks = load_chunks(chunk_size=350, chunk_overlap=50)
vs = FAISS.from_documents(chunks, WordLlamaEmbeddings())
dense = vs.as_retriever(search_kwargs={"k": 3})
bm25 = BM25Retriever.from_documents(chunks, k=3)
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])
print(f"{len(chunks)} parça indekslendi, vektör boyutu {vs.index.d}")

20 parça indekslendi, vektör boyutu 256


In [3]:
soru = "Aday kaydedilmek istemezse ne olur?"
# FAISS burada uzaklık döndürür: küçük değer = daha yakın
for doc, skor in vs.similarity_search_with_score(soru, k=3):
    print(f"{skor:.3f}  {doc.metadata['kaynak']:<30} {doc.metadata.get('bolum')}")

1.082  kvkk_mulakat_kayitlari.md      Aydınlatma ve açık rıza
1.153  aday_degerlendirme_sureci.md   Amaç
1.186  kvkk_mulakat_kayitlari.md      Erişim


In [4]:
import pandas as pd
from rag_eval import retrieval_metrics

def kaynaklar(retriever):
    return lambda q: [d.metadata["kaynak"] for d in retriever.invoke(q)]

pd.DataFrame({ad: retrieval_metrics(kaynaklar(r)) for ad, r in
              [("BM25", bm25), ("Embedding (FAISS)", dense), ("Hibrit", hybrid)]}).T.round(3)

,hit@1,hit@3,MRR
BM25,0.917,0.917,0.917
Embedding (FAISS),0.917,0.917,0.917
Hibrit,0.833,1.000,0.917
